# Atlas integrated scoring

This notebook calculates Levels 1 and 2 of the **Atlas Integrated Structure** for every BSR represented in the five input CSVs.

| Level | Main question | Principal outputs |
|---|---|---|
| **1. Integrated risk** | Where do fish use, limiting-factor condition, biological vulnerability, and population priority overlap? | Life-stage impact and risk, limiting-factor impact and risk, and overall BSR risk |
| **2. Action benefit** | Which action types address the limiting factors contributing most to calculated risk? | Condition improvement, limiting-factor amelioration, and overall action benefit by BSR and action type |

The notebook uses the scores already provided in the standardized CSVs. It does not recreate the source workbooks. In particular, it uses the continuous 0.1-to-1 vulnerability score and the raw LFAT score, defined as directness multiplied by frequency. Level 3 project scoring is not included.

The primary outputs are tidy CSVs in `data/outputs`. If `bsr.gpkg` is present with the CSVs, the notebook also creates `bsr_scores.gpkg` while preserving the source geometry and attributes.


## 1. Inputs, outputs, and the only routine setting

Place the following files in `data/inputs`:

- `Fish Use Scores.csv`
- `LFAT.csv`
- `Limiting factor scores.csv`
- `Population scores.csv`
- `Vulnerability table.csv`
- `bsr.gpkg`, optional spatial input

The automatic file search also accepts a single parenthetically numbered copy of each CSV, such as `Fish Use Scores(8).csv`. If needed, set `INPUT_DIR_OVERRIDE` below to the folder containing the inputs. No other settings are normally required.


In [1]:
from pathlib import Path
import hashlib
import re
import shutil
import sqlite3

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


# Optional: replace None with a folder path if automatic discovery is not appropriate.
INPUT_DIR_OVERRIDE = None

INPUT_STEMS = {
    "fish_use": "Fish Use Scores",
    "lfat": "LFAT",
    "limiting_factor": "Limiting factor scores",
    "population": "Population scores",
    "vulnerability": "Vulnerability table",
}
BSR_INPUT_FILE = "bsr.gpkg"
BSR_OUTPUT_FILE = "bsr_scores.gpkg"


def select_input_file(folder, stem):
    # Return one exact or parenthetically numbered CSV, or None.
    exact = folder / f"{stem}.csv"
    if exact.exists():
        return exact
    matches = sorted(folder.glob(f"{stem}(*).csv"))
    return matches[0] if len(matches) == 1 else None


def candidate_input_directories(start):
    seen = set()
    for folder in (start, *start.parents):
        for candidate in (
            folder / "data" / "inputs",
            folder / "upload",
            folder,
        ):
            resolved = candidate.resolve()
            if resolved not in seen:
                seen.add(resolved)
                yield resolved


def locate_inputs(start, override=None):
    candidates = [Path(override).expanduser().resolve()] if override else list(
        candidate_input_directories(start)
    )
    for folder in candidates:
        if not folder.is_dir():
            continue
        selected = {
            key: select_input_file(folder, stem)
            for key, stem in INPUT_STEMS.items()
        }
        if all(path is not None for path in selected.values()):
            return folder, selected
    expected = ", ".join(f"{stem}.csv" for stem in INPUT_STEMS.values())
    raise FileNotFoundError(
        "Could not find one complete input set. Expected: " + expected
    )


INPUT_DIR, INPUT_PATHS = locate_inputs(
    Path.cwd().resolve(), INPUT_DIR_OVERRIDE
)
if INPUT_DIR.name == "inputs" and INPUT_DIR.parent.name == "data":
    REPO_ROOT = INPUT_DIR.parent.parent
else:
    REPO_ROOT = Path.cwd().resolve()

OUTPUT_DIR = REPO_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BSR_INPUT_PATH = INPUT_DIR / BSR_INPUT_FILE
BSR_OUTPUT_PATH = OUTPUT_DIR / BSR_OUTPUT_FILE
HAS_SPATIAL_INPUT = BSR_INPUT_PATH.exists()

raw = {
    key: pd.read_csv(path)
    for key, path in INPUT_PATHS.items()
}

input_summary = pd.DataFrame(
    [
        {
            "dataset": key,
            "file": path.name,
            "rows": len(raw[key]),
            "columns": len(raw[key].columns),
        }
        for key, path in INPUT_PATHS.items()
    ]
)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Spatial export enabled: {HAS_SPATIAL_INPUT}")
display(input_summary)


Input directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\Atlas_scoring_webapp\data\inputs
Output directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\Atlas_scoring_webapp\data\outputs
Spatial export enabled: True


,dataset,file,rows,columns
0,fish_use,Fish Use Scores.csv,290,12
1,lfat,LFAT.csv,165,19
2,limiting_factor,Limiting factor scores.csv,435,18
3,population,Population scores.csv,20,4
4,vulnerability,Vulnerability table.csv,180,14


## 2. Score fields and transformations

Higher values increase the calculated score. The inputs are used as follows:

| Component | Source field | Scale used | Treatment in this notebook |
|---|---|---:|---|
| Fish use | `fish_use_rating` | 0 to 1 | Used without transformation for each BSR, species, and life stage |
| Population priority | `population_priority` | 0 to 1 | Used without transformation; priorities sum to approximately 1 within each basin and species |
| Limiting-factor condition | `lf_condition_score` | 0.1 to 1 | Used as supplied; the CSV maps its raw 1-to-5 score linearly to 0.1-to-1 |
| Vulnerability | `vulnerability_score` | 0.1 to 1 | Used as supplied; rank 1 maps to 1.0 and rank 15 maps to 0.1 |
| Action relationship | `lfat_score` | 0.01 to 1 | Used as supplied; equals `directness_value × frequency_value` |

The two source columns both named `fish_use_score` are made unambiguous after import. Pandas reads the second as `fish_use_score.1`; the notebook renames them `fish_use_score_raw` and `fish_use_score_normalized`. These BSR-level aggregate fish-use values are retained for comparison but are not used in the integrated equations because the equations use the underlying species and life-stage rows.

Similarly, `corrected_score` and `species_aggregate_score` remain available for auditing but are not used as additional multipliers. Multiplying them again would repeat aggregation already represented by the row-level fish-use and population-priority terms.


In [2]:
EXPECTED_COLUMNS = {
    "fish_use": [
        "bsr", "basin", "bsr_crosswalk_status", "species", "life_stage",
        "fish_use_rating", "corrected_score", "species_aggregate_score",
        "fish_use_score", "rank", "fish_use_score_100", "fish_use_score.1",
    ],
    "lfat": [
        "action_id", "action_type", "action_definition",
        "source_action_label", "limiting_factor",
        "limiting_factor_occurrence", "directness_code",
        "directness_rating", "directness_value", "frequency_code",
        "frequency_rating", "frequency_value", "lfat_score",
        "source_sheet", "source_row", "source_directness_cell",
        "source_frequency_cell", "source_score_cell", "source_notes_cell",
    ],
    "limiting_factor": [
        "bsr", "limiting_factor", "n", "mean_r", "median_r",
        "geo_mean_r", "sd_r", "var_r", "iqr_r", "min_r", "max_r",
        "range_r", "flag_spread", "flag_low_n", "any_flag",
        "lf_condition_score_raw_1_5", "lf_condition_score",
        "condition_transformation",
    ],
    "population": [
        "basin", "species", "life_stage", "population_priority",
    ],
    "vulnerability": [
        "species", "source_life_stage", "life_stage", "limiting_factor",
        "vulnerability_rank", "vulnerability_score", "uncertainty_flag",
        "review_flag", "review_reason", "source_sheet",
        "source_rank_cell", "source_rating_cell", "source_notes_cell",
        "source_uncertainty_cell",
    ],
}

CANONICAL_LF = [
    "Anthropogenic Barriers",
    "Riparian Condition",
    "Floodplain Condition",
    "Side Channel and Wetland Habitat",
    "Channel and Habitat Structure",
    "Decreased Water Quantity",
    "Altered Flow Timing",
    "Decreased Sediment Quantity",
    "Increased Sediment Quantity",
    "Summer Water Temperature",
    "Winter Water Temperature",
    "Water Quality",
    "Predation",
    "Altered Primary Productivity",
    "Non-Native Species Interactions and Competition",
]

schema_rows = []
for key, table in raw.items():
    observed = table.columns.tolist()
    expected = EXPECTED_COLUMNS[key]
    schema_rows.append(
        {
            "dataset": key,
            "observed_columns": len(observed),
            "expected_columns": len(expected),
            "schema_pass": observed == expected,
        }
    )
schema_qc = pd.DataFrame(schema_rows)
if not schema_qc["schema_pass"].all():
    details = {
        key: {
            "observed": raw[key].columns.tolist(),
            "expected": EXPECTED_COLUMNS[key],
        }
        for key in raw
        if raw[key].columns.tolist() != EXPECTED_COLUMNS[key]
    }
    raise ValueError(f"One or more input schemas do not match: {details}")

fish_use = raw["fish_use"].rename(
    columns={
        "fish_use_score": "fish_use_score_raw",
        "fish_use_score.1": "fish_use_score_normalized",
    }
).copy()
population = raw["population"].copy()
condition = raw["limiting_factor"].rename(
    columns={
        "lf_condition_score_raw_1_5": "condition_score_raw_1_5",
        "lf_condition_score": "condition_score",
    }
).copy()
vulnerability = raw["vulnerability"].copy()
lfat = raw["lfat"].copy()

assumptions = pd.DataFrame(
    [
        ["Fish use", "fish_use_rating", "Retained on its source 0-to-1 scale."],
        ["Population priority", "population_priority", "Retained as a decimal proportion."],
        ["Limiting-factor condition", "condition_score", "Source 1-to-5 score already mapped to 0.1-to-1 in the CSV."],
        ["Vulnerability", "vulnerability_score", "Rank 1-to-15 already mapped to 1.0-to-0.1 in the CSV."],
        ["Combined migration", "maximum vulnerability_score", "Uses the higher adult or juvenile migration value without counting migration twice."],
        ["LFAT", "lfat_score", "Raw directness multiplied by frequency; no D/I/N replacement."],
        ["BSR identifiers", "bsr", "The fish-use and limiting-factor files use the same CC1/CC2/etc. identifiers."],
    ],
    columns=["component", "field_or_rule", "implementation"],
)

display(schema_qc)
display(assumptions)


,dataset,observed_columns,expected_columns,schema_pass
0,fish_use,12,12,True
1,lfat,19,19,True
2,limiting_factor,18,18,True
3,population,4,4,True
4,vulnerability,14,14,True


,component,field_or_rule,implementation
0,Fish use,fish_use_rating,Retained on its source 0-to-1 scale.
1,Population priority,population_priority,Retained as a decimal proportion.
2,Limiting-factor condition,condition_score,Source 1-to-5 score already mapped to 0.1-to-1...
3,Vulnerability,vulnerability_score,Rank 1-to-15 already mapped to 1.0-to-0.1 in t...
4,Combined migration,maximum vulnerability_score,Uses the higher adult or juvenile migration va...
5,LFAT,lfat_score,Raw directness multiplied by frequency; no D/I...
6,BSR identifiers,bsr,The fish-use and limiting-factor files use the...


## 3. Align vulnerability life stages

The vulnerability table contains separate `Adult Migration & Holding` and `Juvenile Emigration` rows, while the fish-use and population tables contain one combined `Migration` life stage. The notebook therefore collapses the two source rows to one species × life stage × limiting-factor relationship.

For a combined migration relationship, the value used is:

$$
V_{s,\mathrm{Migration},\ell}
= \max_m\left(V^{\mathrm{source}}_{s,m,\ell}\right)
$$

where:

- $s$ is species;
- $\ell$ is limiting factor;
- $m$ is either the adult or juvenile source migration stage; and
- $V^{\mathrm{source}}$ is the continuous 0.1-to-1 vulnerability score from the CSV.

Taking the maximum represents the more vulnerable migration pathway and avoids adding adult and juvenile migration as if they were two independent fish-use stages. All other life stages have one source row, so their values pass through unchanged.


In [3]:
def any_yes(values):
    is_yes = (
        values.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yes")
    )
    return "Yes" if is_yes.any() else "No"


vulnerability_collapsed = (
    vulnerability.groupby(
        ["species", "life_stage", "limiting_factor"], as_index=False
    )
    .agg(
        vulnerability_rank_min=("vulnerability_rank", "min"),
        vulnerability_rank_max=("vulnerability_rank", "max"),
        vulnerability_score_min=("vulnerability_score", "min"),
        vulnerability_score_max=("vulnerability_score", "max"),
        vulnerability_score=("vulnerability_score", "max"),
        source_vulnerability_rows=("source_life_stage", "size"),
        source_life_stages=(
            "source_life_stage",
            lambda values: " | ".join(sorted(set(values.astype(str)))),
        ),
        vulnerability_review_flag=("review_flag", any_yes),
        uncertainty_notes=(
            "uncertainty_flag",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

display(vulnerability_collapsed.head(10))


,species,life_stage,limiting_factor,vulnerability_rank_min,vulnerability_rank_max,vulnerability_score_min,vulnerability_score_max,vulnerability_score,source_vulnerability_rows,source_life_stages,vulnerability_review_flag,uncertainty_notes
0,Bull Trout,FMO,Altered Flow Timing,11,11,0.357143,0.357143,0.357143,1,FMO (Fluvial),Yes,Unknown Future Climate Change Impacts
1,Bull Trout,FMO,Altered Primary Productivity,12,12,0.292857,0.292857,0.292857,1,FMO (Fluvial),No,
2,Bull Trout,FMO,Anthropogenic Barriers,3,3,0.871429,0.871429,0.871429,1,FMO (Fluvial),No,
3,Bull Trout,FMO,Channel and Habitat Structure,4,4,0.807143,0.807143,0.807143,1,FMO (Fluvial),No,
4,Bull Trout,FMO,Decreased Sediment Quantity,13,13,0.228571,0.228571,0.228571,1,FMO (Fluvial),Yes,Don't know how sediment quantity impacts FMO
5,Bull Trout,FMO,Decreased Water Quantity,2,2,0.935714,0.935714,0.935714,1,FMO (Fluvial),Yes,Unknown Future Climate Change Impacts
6,Bull Trout,FMO,Floodplain Condition,9,9,0.485714,0.485714,0.485714,1,FMO (Fluvial),No,
7,Bull Trout,FMO,Increased Sediment Quantity,14,14,0.164286,0.164286,0.164286,1,FMO (Fluvial),No,
8,Bull Trout,FMO,Non-Native Species Interactions and Competition,10,10,0.421429,0.421429,0.421429,1,FMO (Fluvial),No,
9,Bull Trout,FMO,Predation,7,7,0.614286,0.614286,0.614286,1,FMO (Fluvial),Yes,We don't know the extent of predation. Would b...


## 4. Calculate Level 1 row-level impact and risk

Each fish-use row is expanded across the 15 limiting factors. The calculation unit is one BSR × species × life stage × limiting factor combination.

### Indices

| Symbol | Meaning |
|---|---|
| $b$ | BSR |
| $g(b)$ | Basin containing BSR $b$ |
| $s$ | Fish species |
| $k$ | Life stage |
| $\ell$ | Limiting factor |

### Input variables

| Symbol | Notebook field | Meaning |
|---|---|---|
| $F_{b,s,k}$ | `fish_use_rating` | Fish-use multiplier for BSR $b$, species $s$, and life stage $k$ |
| $C_{b,\ell}$ | `condition_score` | 0.1-to-1 limiting-factor condition multiplier for BSR $b$ and factor $\ell$ |
| $V_{s,k,\ell}$ | `vulnerability_score` | 0.1-to-1 vulnerability of species $s$ and life stage $k$ to factor $\ell$ |
| $P_{g(b),s,k}$ | `population_priority` | Population priority for the basin containing $b$, species $s$, and life stage $k$ |

The row-level impact component is:

$$
I_{b,s,k,\ell}
= F_{b,s,k}\,C_{b,\ell}\,V_{s,k,\ell}
$$

The row-level risk component adds population priority:

$$
R_{b,s,k,\ell}
= I_{b,s,k,\ell}\,P_{g(b),s,k}
= F_{b,s,k}\,C_{b,\ell}\,V_{s,k,\ell}\,P_{g(b),s,k}
$$

A component is zero when fish use is zero. Otherwise, it increases when any contributing multiplier increases. These are prioritization components, not probabilities. They are summed in later steps, so aggregate scores can exceed 1.


In [4]:
fish_population = fish_use.merge(
    population[["basin", "species", "life_stage", "population_priority"]],
    on=["basin", "species", "life_stage"],
    how="left",
    validate="many_to_one",
)

calculation_grid = (
    fish_population.merge(
        vulnerability_collapsed,
        on=["species", "life_stage"],
        how="left",
        validate="many_to_many",
    )
    .merge(
        condition[
            [
                "bsr", "limiting_factor", "condition_score_raw_1_5",
                "condition_score", "n", "any_flag",
            ]
        ],
        on=["bsr", "limiting_factor"],
        how="left",
        validate="many_to_one",
    )
)

calculation_grid["impact_component"] = (
    calculation_grid["fish_use_rating"]
    * calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"]
)
calculation_grid["risk_component"] = (
    calculation_grid["impact_component"]
    * calculation_grid["population_priority"]
)

display(
    calculation_grid[
        [
            "bsr", "species", "life_stage", "limiting_factor",
            "fish_use_rating", "population_priority",
            "condition_score_raw_1_5", "condition_score",
            "vulnerability_rank_min", "vulnerability_rank_max",
            "vulnerability_score", "impact_component", "risk_component",
        ]
    ].head(10)
)


,bsr,species,life_stage,limiting_factor,fish_use_rating,population_priority,condition_score_raw_1_5,condition_score,vulnerability_rank_min,vulnerability_rank_max,vulnerability_score,impact_component,risk_component
0,CC1,Chinook,Spawning,Altered Flow Timing,0.0,0.12,4.31,0.84475,10,10,0.421429,0.0,0.0
1,CC1,Chinook,Spawning,Altered Primary Productivity,0.0,0.12,4.31,0.84475,14,14,0.164286,0.0,0.0
2,CC1,Chinook,Spawning,Anthropogenic Barriers,0.0,0.12,3.68,0.70300,15,15,0.100000,0.0,0.0
3,CC1,Chinook,Spawning,Channel and Habitat Structure,0.0,0.12,4.57,0.90325,5,5,0.742857,0.0,0.0
4,CC1,Chinook,Spawning,Decreased Sediment Quantity,0.0,0.12,2.95,0.53875,3,3,0.871429,0.0,0.0
5,CC1,Chinook,Spawning,Decreased Water Quantity,0.0,0.12,5.00,1.00000,2,2,0.935714,0.0,0.0
6,CC1,Chinook,Spawning,Floodplain Condition,0.0,0.12,4.78,0.95050,6,6,0.678571,0.0,0.0
7,CC1,Chinook,Spawning,Increased Sediment Quantity,0.0,0.12,4.51,0.88975,11,11,0.357143,0.0,0.0
8,CC1,Chinook,Spawning,Non-Native Species Interactions and Competition,0.0,0.12,4.64,0.91900,12,12,0.292857,0.0,0.0
9,CC1,Chinook,Spawning,Predation,0.0,0.12,4.22,0.82450,9,9,0.485714,0.0,0.0


## 5. Aggregate Level 1 scores

The same row-level grid is summarized in three ways. No additional weights or averages are introduced.

### Species and life-stage summaries

For each BSR, species, and life stage:

$$
I^{\mathrm{life}}_{b,s,k} = \sum_{\ell} I_{b,s,k,\ell}
\qquad
R^{\mathrm{life}}_{b,s,k} = \sum_{\ell} R_{b,s,k,\ell}
$$

### Limiting-factor summaries

For each BSR and limiting factor:

$$
I^{\mathrm{LF}}_{b,\ell} = \sum_s\sum_k I_{b,s,k,\ell}
\qquad
R^{\mathrm{LF}}_{b,\ell} = \sum_s\sum_k R_{b,s,k,\ell}
$$

### Overall BSR summaries

The overall BSR impact and risk scores are:

$$
I^{\mathrm{overall}}_b = \sum_s\sum_k\sum_{\ell} I_{b,s,k,\ell}
$$

$$
R^{\mathrm{overall}}_b = \sum_s\sum_k\sum_{\ell} R_{b,s,k,\ell}
$$

The life-stage and limiting-factor summaries partition the same grid, so summing either summary must reproduce the overall BSR values. Rankings are calculated within each BSR. Rank 1 identifies the largest contribution, and tied top contributors are retained together in the BSR summary.


In [5]:
life_stage_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "species", "life_stage"], as_index=False
    )
    .agg(
        fish_use_rating=("fish_use_rating", "first"),
        population_priority=("population_priority", "first"),
        impact_score=("impact_component", "sum"),
        risk_score=("risk_component", "sum"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
    )
)
life_stage_scores["risk_rank_within_bsr"] = (
    life_stage_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
life_stage_scores["priority_label"] = (
    life_stage_scores["species"] + " | " + life_stage_scores["life_stage"]
)

limiting_factor_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "limiting_factor"], as_index=False
    )
    .agg(
        condition_score_raw_1_5=("condition_score_raw_1_5", "first"),
        condition_score=("condition_score", "first"),
        condition_rating_n=("n", "first"),
        condition_review_flag=("any_flag", "first"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
        impact_score=("impact_component", "sum"),
        risk_score=("risk_component", "sum"),
    )
)
limiting_factor_scores["risk_rank_within_bsr"] = (
    limiting_factor_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

source_fish_use = (
    fish_use.groupby(["bsr", "basin"], as_index=False)
    .agg(
        source_fish_use_score_raw=("fish_use_score_raw", "first"),
        source_fish_use_score_normalized=(
            "fish_use_score_normalized", "first"
        ),
        source_fish_use_score_100=("fish_use_score_100", "first"),
        raw_score_variants=("fish_use_score_raw", "nunique"),
        normalized_score_variants=(
            "fish_use_score_normalized", "nunique"
        ),
    )
)

bsr_from_life_stage = (
    life_stage_scores.groupby(["bsr", "basin"], as_index=False)
    .agg(
        detailed_fish_use_score=("fish_use_rating", "sum"),
        overall_impact_score=("impact_score", "sum"),
        overall_risk_score=("risk_score", "sum"),
    )
)
bsr_from_limiting_factor = (
    limiting_factor_scores.groupby("bsr", as_index=False)
    .agg(
        lf_sum_impact_score=("impact_score", "sum"),
        lf_sum_risk_score=("risk_score", "sum"),
    )
)

top_life_stage = (
    life_stage_scores.loc[life_stage_scores["risk_rank_within_bsr"].eq(1)]
    .groupby("bsr", as_index=False)
    .agg(
        highest_priority_life_stage=(
            "priority_label", lambda values: "; ".join(sorted(values))
        ),
        top_life_stage_risk_score=("risk_score", "first"),
        highest_priority_life_stage_tie_count=("priority_label", "size"),
    )
)
top_limiting_factor = (
    limiting_factor_scores.loc[
        limiting_factor_scores["risk_rank_within_bsr"].eq(1)
    ]
    .groupby("bsr", as_index=False)
    .agg(
        highest_priority_limiting_factor=(
            "limiting_factor", lambda values: "; ".join(sorted(values))
        ),
        top_limiting_factor_risk_score=("risk_score", "first"),
        highest_priority_limiting_factor_tie_count=(
            "limiting_factor", "size"
        ),
    )
)

bsr_scores = (
    bsr_from_life_stage
    .merge(bsr_from_limiting_factor, on="bsr", validate="one_to_one")
    .merge(source_fish_use, on=["bsr", "basin"], validate="one_to_one")
    .merge(top_life_stage, on="bsr", validate="one_to_one")
    .merge(top_limiting_factor, on="bsr", validate="one_to_one")
)
bsr_scores["impact_balance_difference"] = (
    bsr_scores["overall_impact_score"]
    - bsr_scores["lf_sum_impact_score"]
)
bsr_scores["risk_balance_difference"] = (
    bsr_scores["overall_risk_score"]
    - bsr_scores["lf_sum_risk_score"]
)
bsr_scores["final_score"] = bsr_scores["overall_risk_score"]

display(
    bsr_scores[
        [
            "bsr", "overall_risk_score", "highest_priority_life_stage",
            "highest_priority_limiting_factor",
            "source_fish_use_score_normalized",
            "detailed_fish_use_score",
        ]
    ].sort_values("overall_risk_score", ascending=False).head(10)
)


,bsr,overall_risk_score,highest_priority_life_stage,highest_priority_limiting_factor,source_fish_use_score_normalized,detailed_fish_use_score
17,UGR17,8.342559,Bull Trout | FMO,Channel and Habitat Structure,0.6039,6.58
24,UGR5,8.273506,Chinook | Migration,Channel and Habitat Structure,0.5242,3.43
5,CC6,7.874337,Bull Trout | FMO,Decreased Water Quantity,0.6329,6.54
22,UGR3,7.279930,Chinook | Migration,Decreased Water Quantity,0.6514,4.00
21,UGR20,6.963120,Bull Trout | Spawning & Resident,Non-Native Species Interactions and Competition,0.8697,7.08
4,CC5,6.876646,Chinook | Winter Rearing,Channel and Habitat Structure,0.4663,4.56
15,UGR15,6.620845,Bull Trout | FMO,Channel and Habitat Structure,0.6126,6.37
6,CC7,6.598783,Bull Trout | FMO,Summer Water Temperature,1.0000,7.82
19,UGR19,6.543027,Bull Trout | Spawning & Resident,Riparian Condition,0.7778,6.45
11,UGR11,6.310657,Chinook | Winter Rearing,Channel and Habitat Structure,0.4504,4.11


## 6. Calculate Level 2 action-specific scores

Level 2 applies the LFAT relationship between each action type and limiting factor to the Level 1 results.

### Additional index and variable

| Symbol | Meaning |
|---|---|
| $a$ | Action type |
| $W_{a,\ell}$ | Raw `lfat_score` for action $a$ and limiting factor $\ell$ |

The LFAT weight is supplied by the CSV as:

$$
W_{a,\ell} = D_{a,\ell}\,Q_{a,\ell}
$$

where $D_{a,\ell}$ is `directness_value` and $Q_{a,\ell}$ is `frequency_value`. The symbol $Q$ is used here to avoid confusing the frequency multiplier with fish use, $F$.

Three action-specific summaries are then calculated for each BSR:

$$
CI_{b,a} = \sum_{\ell} W_{a,\ell}\,C_{b,\ell}
$$

$$
LA_{b,a} = \sum_{\ell} W_{a,\ell}\,I^{\mathrm{LF}}_{b,\ell}
$$

$$
B_{b,a} = \sum_{\ell} W_{a,\ell}\,R^{\mathrm{LF}}_{b,\ell}
$$

where:

- $CI_{b,a}$ is `condition_improvement_score`, based only on condition and the action crosswalk;
- $LA_{b,a}$ is `limiting_factor_amelioration_score`, which also incorporates fish use and vulnerability through $I^{\mathrm{LF}}$; and
- $B_{b,a}$ is `overall_benefit_score`, which additionally incorporates population priority through $R^{\mathrm{LF}}$.

These scores represent relative alignment with the scored limiting factors. They do not estimate the realized benefit, cost, or feasibility of a specific project.


In [6]:
action_components = limiting_factor_scores.merge(
    lfat[
        [
            "action_id", "action_type", "action_definition",
            "limiting_factor", "directness_code", "directness_value",
            "frequency_code", "frequency_value", "lfat_score",
        ]
    ],
    on="limiting_factor",
    how="inner",
    validate="many_to_many",
)
action_components["condition_improvement_component"] = (
    action_components["condition_score"]
    * action_components["lfat_score"]
)
action_components["amelioration_component"] = (
    action_components["impact_score"]
    * action_components["lfat_score"]
)
action_components["benefit_component"] = (
    action_components["risk_score"]
    * action_components["lfat_score"]
)

action_scores = (
    action_components.groupby(
        [
            "bsr", "basin", "action_id", "action_type",
            "action_definition",
        ],
        as_index=False,
    )
    .agg(
        condition_improvement_score=(
            "condition_improvement_component", "sum"
        ),
        limiting_factor_amelioration_score=(
            "amelioration_component", "sum"
        ),
        overall_benefit_score=("benefit_component", "sum"),
        condition_review_count=(
            "condition_review_flag",
            lambda values: int(
                values.fillna(False)
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["true", "yes", "1"])
                .sum()
            ),
        ),
        vulnerability_review_count=(
            "vulnerability_review_flag",
            lambda values: int(values.eq("Yes").sum()),
        ),
    )
)
action_scores["benefit_rank_within_bsr"] = (
    action_scores.groupby("bsr")["overall_benefit_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

top_action = (
    action_scores.loc[action_scores["benefit_rank_within_bsr"].eq(1)]
    .assign(
        priority_action_label=lambda table: (
            table["action_id"].astype(str) + " | " + table["action_type"]
        )
    )
    .groupby("bsr", as_index=False)
    .agg(
        highest_priority_action=(
            "priority_action_label",
            lambda values: "; ".join(sorted(values)),
        ),
        highest_priority_action_benefit_score=(
            "overall_benefit_score", "first"
        ),
        highest_priority_action_tie_count=(
            "priority_action_label", "size"
        ),
    )
)
provisional_action_sum = (
    action_scores.groupby("bsr", as_index=False)["overall_benefit_score"]
    .sum()
    .rename(
        columns={
            "overall_benefit_score": "sum_action_benefit_provisional"
        }
    )
)
bsr_scores = (
    bsr_scores
    .merge(top_action, on="bsr", validate="one_to_one")
    .merge(provisional_action_sum, on="bsr", validate="one_to_one")
)

display(
    action_scores[
        [
            "bsr", "action_type", "condition_improvement_score",
            "limiting_factor_amelioration_score", "overall_benefit_score",
            "benefit_rank_within_bsr",
        ]
    ].sort_values(["bsr", "benefit_rank_within_bsr"]).head(15)
)


,bsr,action_type,condition_improvement_score,limiting_factor_amelioration_score,overall_benefit_score,benefit_rank_within_bsr
2,CC1,Floodplain – Reconnect and Restore,5.662038,8.649735,2.923958,1
3,CC1,Riparian Vegetation Restoration,5.059518,7.443369,2.519184,2
1,CC1,Instream Complexity Improvement,3.908902,6.508857,2.201480,3
5,CC1,Instream Flow Restoration,3.933810,5.953257,2.036803,4
0,CC1,Protect Land (Easement and Acquisition),2.739425,4.145717,1.403299,5
6,CC1,"Thermal Refuge Enhancement (reconnect, expand)",2.086215,2.949860,1.007867,6
4,CC1,Fish Passage – Barrier Removal and Replacement,1.951982,2.873153,0.977435,7
7,CC1,Water Quality Improvement (not including tempe...,1.776323,2.795095,0.946548,8
10,CC1,Species Management (non-native or unnatural),1.853948,2.206437,0.740618,9
9,CC1,Upland Treatments,1.576763,2.025256,0.688122,10


## 7. Save tidy results and, when available, the scored GeoPackage

The detailed output tables retain the source and intermediate fields needed to audit the calculations. `final_score` is an explicit alias of `overall_risk_score` for downstream mapping applications.

If `bsr.gpkg` is available, its `BSR` field is matched directly to the standardized `bsr` field. The notebook copies the GeoPackage, preserves its geometry and existing attributes, and adds the BSR summary fields. If the GeoPackage is absent, all CSV calculations and quality-control checks still run.


In [7]:
identifier_review = (
    fish_use[["bsr", "basin", "bsr_crosswalk_status"]]
    .drop_duplicates()
    .sort_values(["basin", "bsr"])
    .reset_index(drop=True)
)

outputs = {
    "calculation_grid.csv": calculation_grid,
    "life_stage_scores.csv": life_stage_scores,
    "limiting_factor_scores_integrated.csv": limiting_factor_scores,
    "bsr_scores.csv": bsr_scores,
    "action_scores.csv": action_scores,
    "action_score_components.csv": action_components,
    "assumptions_for_review.csv": assumptions,
    "bsr_identifiers_for_review.csv": identifier_review,
    "vulnerability_scores_for_review.csv": vulnerability_collapsed,
}

for filename, table in outputs.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)


def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


def sqlite_type(series):
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    if pd.api.types.is_numeric_dtype(series):
        return "REAL"
    return "TEXT"


def sqlite_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def geometry_digest(path, feature_table, key_field, geometry_field):
    digest = hashlib.sha256()
    query = (
        f"SELECT {quote_identifier(key_field)}, "
        f"{quote_identifier(geometry_field)} "
        f"FROM {quote_identifier(feature_table)} "
        f"ORDER BY {quote_identifier(key_field)}"
    )
    with sqlite3.connect(path) as connection:
        for key, geometry in connection.execute(query):
            digest.update(str(key).encode("utf-8"))
            digest.update(bytes(geometry) if geometry is not None else b"")
    return digest.hexdigest()


def write_scored_bsr_gpkg(input_path, output_path, summary):
    with sqlite3.connect(input_path) as connection:
        feature_tables = [
            row[0]
            for row in connection.execute(
                "SELECT table_name FROM gpkg_contents WHERE data_type = 'features'"
            )
        ]
        candidates = []
        for table_name in feature_tables:
            columns = [
                row[1]
                for row in connection.execute(
                    f"PRAGMA table_info({quote_identifier(table_name)})"
                )
            ]
            source_key = next(
                (column for column in columns if column.lower() == "bsr"),
                None,
            )
            if source_key is not None:
                candidates.append((table_name, source_key, columns))

        if len(candidates) != 1:
            raise ValueError(
                "Expected exactly one feature layer containing a BSR field; "
                f"found {len(candidates)}."
            )
        feature_table, source_key, source_columns = candidates[0]
        geometry_row = connection.execute(
            "SELECT column_name FROM gpkg_geometry_columns WHERE table_name = ?",
            (feature_table,),
        ).fetchone()
        if geometry_row is None:
            raise ValueError(
                f"No geometry field is registered for layer {feature_table}."
            )
        geometry_field = geometry_row[0]
        spatial_keys = {
            str(row[0]).strip()
            for row in connection.execute(
                f"SELECT {quote_identifier(source_key)} "
                f"FROM {quote_identifier(feature_table)}"
            )
        }

    summary_keys = set(summary["bsr"].astype(str).str.strip())
    if spatial_keys != summary_keys:
        raise ValueError(
            "The GeoPackage and score summary have different BSR coverage. "
            f"Missing scores: {sorted(spatial_keys - summary_keys)}; "
            f"missing geometry: {sorted(summary_keys - spatial_keys)}"
        )

    scored = summary.rename(columns={"bsr": "score_bsr"}).copy()
    scored_columns = scored.columns.tolist()
    existing_lower = {column.lower() for column in source_columns}
    collisions = [
        column for column in scored_columns if column.lower() in existing_lower
    ]
    if collisions:
        raise ValueError(
            "Score fields collide with existing GeoPackage fields: "
            + ", ".join(collisions)
        )

    temporary_path = output_path.with_name(f".{output_path.stem}.tmp.gpkg")
    if temporary_path.exists():
        temporary_path.unlink()
    shutil.copy2(input_path, temporary_path)

    try:
        with sqlite3.connect(temporary_path) as connection:
            # Some GeoPackages contain RTree update triggers that invoke these
            # functions even when only non-geometry fields are updated.
            connection.create_function(
                "ST_IsEmpty", 1, lambda geometry: 1 if geometry is None else 0
            )
            for function_name in ("ST_MinX", "ST_MaxX", "ST_MinY", "ST_MaxY"):
                connection.create_function(
                    function_name, 1, lambda geometry: 0.0
                )

            for column in scored_columns:
                connection.execute(
                    f"ALTER TABLE {quote_identifier(feature_table)} "
                    f"ADD COLUMN {quote_identifier(column)} "
                    f"{sqlite_type(scored[column])}"
                )

            assignments = ", ".join(
                f"{quote_identifier(column)} = ?" for column in scored_columns
            )
            update_sql = (
                f"UPDATE {quote_identifier(feature_table)} SET {assignments} "
                f"WHERE TRIM(CAST({quote_identifier(source_key)} AS TEXT)) = ?"
            )
            for original_bsr, (_, row) in zip(
                summary["bsr"].astype(str), scored.iterrows()
            ):
                values = [sqlite_value(row[column]) for column in scored_columns]
                values.append(original_bsr.strip())
                cursor = connection.execute(update_sql, values)
                if cursor.rowcount != 1:
                    raise ValueError(
                        f"Expected one spatial row for {original_bsr}; "
                        f"updated {cursor.rowcount}."
                    )

            connection.execute(
                "UPDATE gpkg_contents SET identifier = ?, description = ? "
                "WHERE table_name = ?",
                (
                    "Atlas scored BSRs",
                    "BSR geometry with Level 1 and Level 2 score summaries",
                    feature_table,
                ),
            )
            integrity = connection.execute(
                "PRAGMA integrity_check"
            ).fetchone()[0]
            if integrity != "ok":
                raise ValueError(
                    f"GeoPackage integrity check failed: {integrity}"
                )

        if output_path.exists():
            output_path.unlink()
        temporary_path.replace(output_path)
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    return feature_table, geometry_field, source_key, scored_columns


SPATIAL_EXPORT_CREATED = False
spatial_metadata = {}
if HAS_SPATIAL_INPUT:
    (
        BSR_OUTPUT_LAYER,
        BSR_GEOMETRY_FIELD,
        BSR_SOURCE_KEY,
        BSR_SCORE_FIELDS,
    ) = write_scored_bsr_gpkg(
        BSR_INPUT_PATH, BSR_OUTPUT_PATH, bsr_scores
    )
    SPATIAL_EXPORT_CREATED = True
    spatial_metadata = {
        "layer": BSR_OUTPUT_LAYER,
        "geometry_field": BSR_GEOMETRY_FIELD,
        "source_key": BSR_SOURCE_KEY,
    }
else:
    print(
        "No bsr.gpkg was found. CSV outputs were created; spatial export was skipped."
    )

manifest_rows = [
    {"file": filename, "rows": len(table), "path": OUTPUT_DIR / filename}
    for filename, table in outputs.items()
]
if SPATIAL_EXPORT_CREATED:
    manifest_rows.append(
        {
            "file": BSR_OUTPUT_FILE,
            "rows": len(bsr_scores),
            "path": BSR_OUTPUT_PATH,
        }
    )
output_manifest = pd.DataFrame(manifest_rows)
display(output_manifest)


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\AlexThornton-Dunwood\\OneDrive - Lichen Land & Water\\Documents\\GitHub\\Atlas_scoring_webapp\\data\\outputs\\.bsr_scores.tmp.gpkg'

## 8. Interpretation and publication cautions

- `final_score` is identical to `overall_risk_score`. It is a relative Level 1 prioritization score, not a probability or measured magnitude of ecological harm.
- Aggregate scores are sums across multiple species, life stages, and limiting factors. They are not constrained to 0-to-1 even though each input multiplier is bounded.
- BSRs are directly comparable here because the quality-control checks require the same species × life-stage and limiting-factor coverage for every BSR.
- A high `overall_benefit_score` indicates that an action type aligns with limiting factors contributing to calculated risk. It does not account for project feasibility, cost, landowner willingness, implementation constraints, or site-specific effectiveness.
- `sum_action_benefit_provisional` is exploratory. Summing across action types can double count the same limiting-factor pathway and should not be interpreted as an additive estimate of realized restoration benefit.
- The maximum adult/juvenile vulnerability used for combined migration is a conservative aggregation choice. The detailed review table retains the source stages and score range.
- Review rows identified by the source uncertainty and quality-control flags before publication. Also confirm any BSRs whose `bsr_crosswalk_status` remains provisional.


## 9. Quality control

The checks below verify the attached schemas, score transformations, key uniqueness, coverage of all 15 limiting factors, completeness of every join, reconciliation of alternative aggregation paths, saved outputs, and optional GeoPackage integrity and geometry preservation. An assertion stops the notebook at the first failed requirement rather than allowing incomplete scores to propagate.


In [ ]:
# QC 1: source values and transformations
required_non_null = {
    "fish-use BSR": fish_use["bsr"],
    "fish-use rating": fish_use["fish_use_rating"],
    "population priority": population["population_priority"],
    "condition raw score": condition["condition_score_raw_1_5"],
    "condition score": condition["condition_score"],
    "vulnerability rank": vulnerability["vulnerability_rank"],
    "vulnerability score": vulnerability["vulnerability_score"],
    "LFAT directness": lfat["directness_value"],
    "LFAT frequency": lfat["frequency_value"],
    "LFAT score": lfat["lfat_score"],
}
unresolved = {
    name: int(series.isna().sum())
    for name, series in required_non_null.items()
}
assert not any(unresolved.values()), unresolved

assert fish_use["fish_use_rating"].between(0, 1).all()
assert fish_use["fish_use_score_normalized"].between(0, 1).all()
assert np.allclose(
    fish_use["fish_use_score_normalized"],
    fish_use["fish_use_score_100"] / 100,
    atol=1e-12,
)
assert population["population_priority"].between(0, 1).all()

population_sums = (
    population.groupby(["basin", "species"])["population_priority"].sum()
)
assert np.allclose(population_sums, 1.0, atol=0.011)

expected_condition_score = 0.10 + (
    condition["condition_score_raw_1_5"] - 1.0
) * (0.90 / 4.0)
assert condition["condition_score_raw_1_5"].between(1, 5).all()
assert condition["condition_score"].between(0.10, 1.00).all()
assert np.allclose(
    condition["condition_score"], expected_condition_score, atol=1e-12
)

expected_vulnerability_score = 1.0 - 0.9 * (
    vulnerability["vulnerability_rank"] - 1.0
) / 14.0
assert vulnerability["vulnerability_rank"].between(1, 15).all()
assert vulnerability["vulnerability_score"].between(0.10, 1.00).all()
assert np.allclose(
    vulnerability["vulnerability_score"],
    expected_vulnerability_score,
    atol=1e-8,
)

vulnerability_source_coverage = vulnerability.groupby(
    ["species", "source_life_stage"]
).agg(
    rows=("limiting_factor", "size"),
    limiting_factors=("limiting_factor", "nunique"),
)
# Tied source ranks are valid. Every source stage must still contain one
# record for each of the 15 limiting factors.
assert vulnerability_source_coverage["rows"].eq(15).all()
assert vulnerability_source_coverage["limiting_factors"].eq(15).all()

assert np.allclose(
    lfat["lfat_score"],
    lfat["directness_value"] * lfat["frequency_value"],
    atol=1e-12,
)
assert lfat["lfat_score"].between(0, 1).all()

display(population_sums.rename("priority_sum").reset_index())
display(pd.DataFrame([unresolved]).T.rename(columns={0: "missing_values"}))


In [ ]:
# QC 2: keys, coverage, joins, balances, and saved outputs
assert not fish_use.duplicated(["bsr", "species", "life_stage"]).any()
assert not population.duplicated(["basin", "species", "life_stage"]).any()
assert not condition.duplicated(["bsr", "limiting_factor"]).any()
assert not vulnerability.duplicated(
    ["species", "source_life_stage", "limiting_factor"]
).any()
assert not lfat.duplicated(["action_id", "limiting_factor"]).any()
assert not vulnerability_collapsed.duplicated(
    ["species", "life_stage", "limiting_factor"]
).any()

canonical_lf_set = set(CANONICAL_LF)
assert set(condition["limiting_factor"]) == canonical_lf_set
assert set(vulnerability["limiting_factor"]) == canonical_lf_set
assert set(lfat["limiting_factor"]) == canonical_lf_set
assert set(fish_use["bsr"]) == set(condition["bsr"])
assert condition.groupby("bsr")["limiting_factor"].nunique().eq(15).all()
assert lfat.groupby("action_id")["limiting_factor"].nunique().eq(15).all()

fish_population_keys = set(
    map(
        tuple,
        fish_use[["basin", "species", "life_stage"]]
        .drop_duplicates()
        .to_numpy(),
    )
)
population_keys = set(
    map(tuple, population[["basin", "species", "life_stage"]].to_numpy())
)
assert fish_population_keys == population_keys

fish_stage_keys = set(
    map(
        tuple,
        fish_use[["species", "life_stage"]].drop_duplicates().to_numpy(),
    )
)
vulnerability_stage_keys = set(
    map(
        tuple,
        vulnerability_collapsed[["species", "life_stage"]]
        .drop_duplicates()
        .to_numpy(),
    )
)
assert fish_stage_keys == vulnerability_stage_keys

expected_vulnerability_rows = len(fish_stage_keys) * len(CANONICAL_LF)
expected_grid_rows = len(fish_use) * len(CANONICAL_LF)
required_grid_fields = [
    "fish_use_rating", "population_priority", "vulnerability_score",
    "condition_score", "impact_component", "risk_component",
]
assert len(vulnerability_collapsed) == expected_vulnerability_rows
assert len(calculation_grid) == expected_grid_rows
assert not calculation_grid[required_grid_fields].isna().any().any()
assert calculation_grid.groupby("bsr").size().eq(
    len(fish_stage_keys) * len(CANONICAL_LF)
).all()
assert source_fish_use["raw_score_variants"].eq(1).all()
assert source_fish_use["normalized_score_variants"].eq(1).all()
assert np.allclose(bsr_scores["impact_balance_difference"], 0.0)
assert np.allclose(bsr_scores["risk_balance_difference"], 0.0)
assert np.allclose(
    bsr_scores["final_score"], bsr_scores["overall_risk_score"]
)
assert all((OUTPUT_DIR / filename).exists() for filename in outputs)

qc_rows = [
    ["Fish-use BSRs", fish_use["bsr"].nunique(), condition["bsr"].nunique()],
    ["Condition BSRs", condition["bsr"].nunique(), fish_use["bsr"].nunique()],
    ["Canonical limiting factors", len(CANONICAL_LF), 15],
    ["Collapsed vulnerability relationships", len(vulnerability_collapsed), expected_vulnerability_rows],
    ["Level 1 calculation-grid rows", len(calculation_grid), expected_grid_rows],
    ["LFAT action-factor relationships", len(lfat), lfat["action_id"].nunique() * len(CANONICAL_LF)],
    ["Saved CSV tables", sum((OUTPUT_DIR / filename).exists() for filename in outputs), len(outputs)],
]

if SPATIAL_EXPORT_CREATED:
    assert BSR_OUTPUT_PATH.exists()
    with sqlite3.connect(BSR_OUTPUT_PATH) as connection:
        scored_row_count = connection.execute(
            f"SELECT COUNT(*) FROM {quote_identifier(BSR_OUTPUT_LAYER)}"
        ).fetchone()[0]
        scored_null_count = connection.execute(
            f"SELECT COUNT(*) FROM {quote_identifier(BSR_OUTPUT_LAYER)} "
            "WHERE score_bsr IS NULL OR final_score IS NULL "
            "OR overall_risk_score IS NULL "
            f"OR {quote_identifier(BSR_GEOMETRY_FIELD)} IS NULL"
        ).fetchone()[0]
        final_score_difference = connection.execute(
            "SELECT MAX(ABS(final_score - overall_risk_score)) "
            f"FROM {quote_identifier(BSR_OUTPUT_LAYER)}"
        ).fetchone()[0]
        scored_integrity = connection.execute(
            "PRAGMA integrity_check"
        ).fetchone()[0]

    assert scored_row_count == len(bsr_scores)
    assert scored_null_count == 0
    assert final_score_difference <= 1e-12
    assert scored_integrity == "ok"
    assert geometry_digest(
        BSR_INPUT_PATH,
        BSR_OUTPUT_LAYER,
        BSR_SOURCE_KEY,
        BSR_GEOMETRY_FIELD,
    ) == geometry_digest(
        BSR_OUTPUT_PATH,
        BSR_OUTPUT_LAYER,
        BSR_SOURCE_KEY,
        BSR_GEOMETRY_FIELD,
    )
    qc_rows.extend(
        [
            ["Scored GeoPackage BSR rows", scored_row_count, len(bsr_scores)],
            ["Scored GeoPackage null score/geometry rows", scored_null_count, 0],
        ]
    )

qc_summary = pd.DataFrame(
    qc_rows, columns=["check", "observed", "expected"]
)
qc_summary["pass"] = qc_summary["observed"].eq(qc_summary["expected"])
assert qc_summary["pass"].all()

display(qc_summary)
display(
    identifier_review.query("bsr_crosswalk_status != 'exact_identifier'")
)


### QC 3. Hand-calculated example

This small example uses one hypothetical BSR, two life stages, two limiting factors, and two actions. It verifies that the programmed aggregation reproduces independently specified expected values. The example values test the equations only; they do not replace or recalibrate the input CSV scores.


In [ ]:
test_population = {"Spawning": 0.60, "Rearing": 0.40}
test_fish_use = {"Spawning": 1.00, "Rearing": 1.00}
test_condition = {"Temperature": 0.10, "Instream Complexity": 1.00}
test_vulnerability = {
    ("Spawning", "Temperature"): 1.00,
    ("Rearing", "Temperature"): 0.50,
    ("Spawning", "Instream Complexity"): 0.50,
    ("Rearing", "Instream Complexity"): 1.00,
}
test_action_weight = {
    ("Floodplain Restoration", "Temperature"): 0.50,
    ("Floodplain Restoration", "Instream Complexity"): 1.00,
    ("Riparian Planting", "Temperature"): 1.00,
    ("Riparian Planting", "Instream Complexity"): 0.50,
}

test_rows = []
for life_stage, fish_score in test_fish_use.items():
    for limiting_factor, condition_score in test_condition.items():
        impact = (
            fish_score
            * condition_score
            * test_vulnerability[(life_stage, limiting_factor)]
        )
        test_rows.append(
            {
                "life_stage": life_stage,
                "limiting_factor": limiting_factor,
                "condition_score": condition_score,
                "impact_component": impact,
                "risk_component": impact * test_population[life_stage],
            }
        )
test_grid = pd.DataFrame(test_rows)

test_life = test_grid.groupby("life_stage", as_index=False).agg(
    impact_score=("impact_component", "sum"),
    risk_score=("risk_component", "sum"),
)
test_lf = test_grid.groupby("limiting_factor", as_index=False).agg(
    condition_score=("condition_score", "first"),
    impact_score=("impact_component", "sum"),
    risk_score=("risk_component", "sum"),
)

test_actions = []
for action in ["Floodplain Restoration", "Riparian Planting"]:
    rows = test_lf.copy()
    rows["weight"] = rows["limiting_factor"].map(
        lambda factor: test_action_weight[(action, factor)]
    )
    test_actions.append(
        {
            "action": action,
            "condition_improvement_score": (
                rows["condition_score"] * rows["weight"]
            ).sum(),
            "limiting_factor_amelioration_score": (
                rows["impact_score"] * rows["weight"]
            ).sum(),
            "overall_benefit_score": (
                rows["risk_score"] * rows["weight"]
            ).sum(),
        }
    )
test_actions = pd.DataFrame(test_actions)

expected_life = {
    "Spawning": (0.60, 0.36),
    "Rearing": (1.05, 0.42),
}
for row in test_life.itertuples(index=False):
    expected_impact, expected_risk = expected_life[row.life_stage]
    assert np.isclose(row.impact_score, expected_impact)
    assert np.isclose(row.risk_score, expected_risk)

assert np.isclose(test_lf["impact_score"].sum(), 1.65)
assert np.isclose(test_lf["risk_score"].sum(), 0.78)
assert np.allclose(
    test_actions["condition_improvement_score"], [1.05, 0.60]
)
assert np.allclose(
    test_actions["limiting_factor_amelioration_score"], [1.575, 0.90]
)
assert np.allclose(test_actions["overall_benefit_score"], [0.74, 0.43])
assert np.isclose(test_actions["overall_benefit_score"].sum(), 1.17)

print("All schema, transformation, join, balance, output, and equation checks passed.")
display(test_life)
display(test_lf)
display(test_actions)
